# Louvain community detection - Art Institute of Chicago

This notebook builds a cross-author graph from cosine similarity between artwork feature vectors, then runs Louvain directly on that graph. `department_title` is used only for external evaluation after communities are found; it is never used to create edges. Louvain chooses the number of communities from graph structure rather than forcing it to equal the number of departments.

On Kaggle, add the `artic_department_25000` data folder via **Add Input**. It may be at the Input root or inside `Database/artic_department_25000`. The notebook detects the path and saves results to `/kaggle/working/artic_department_25000/louvain_results/`. Change `N_NEIGHBORS`, `RUNS`, `SEED`, or `RESOLUTION` in the Louvain cell if needed.


In [ ]:
from pathlib import Path
import os

DATASET_NAME = "artic_department_25000"
REQUIRED_FILES = ("features.txt", "nodes.csv", "metadata.json")


def has_dataset_files(path):
    return path.is_dir() and all((path / filename).is_file() for filename in REQUIRED_FILES)


kaggle_input = Path("/kaggle/input")
kaggle_working = Path("/kaggle/working")
is_kaggle = kaggle_working.is_dir()
search_roots = [Path.cwd(), *Path.cwd().parents]
if kaggle_input.is_dir():
    search_roots.extend((kaggle_input, *(p for p in kaggle_input.iterdir() if p.is_dir())))

candidates = []
for root in search_roots:
    candidates.extend((
        root / "Database" / DATASET_NAME,
        root / DATASET_NAME,
        root / "CDNMF" / "Database" / DATASET_NAME,
    ))
    if root.name == DATASET_NAME:
        candidates.append(root)

DATASET_DIR = next((path.resolve() for path in candidates if has_dataset_files(path)), None)
if DATASET_DIR is None:
    raise FileNotFoundError(
        f"Could not find {DATASET_NAME} with edges.txt, nodes.csv, and metadata.json. "
        "Add the dataset folder as a Kaggle Input."
    )

if is_kaggle:
    OUTPUT_DIR = kaggle_working / DATASET_NAME / "louvain_results"
else:
    OUTPUT_DIR = DATASET_DIR / "louvain_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
import csv
import json
import time

import networkx as nx
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)
from sklearn.neighbors import NearestNeighbors

with (DATASET_DIR / "metadata.json").open(encoding="utf-8") as source:
    metadata = json.load(source)
if metadata.get("label_field") != "department_title":
    raise ValueError("Expected department_title as the external evaluation label.")

with (DATASET_DIR / "nodes.csv").open(encoding="utf-8-sig", newline="") as source:
    nodes = list(csv.DictReader(source))
if len(nodes) != int(metadata["processed_artworks"]):
    raise ValueError("nodes.csv row count does not match metadata.")
if [int(row["node_index"]) for row in nodes] != list(range(len(nodes))):
    raise ValueError("node_index in nodes.csv must be contiguous and ordered.")

node_ids = [row["artwork_id"] for row in nodes]
node_id_to_index = {node_id: index for index, node_id in enumerate(node_ids)}
if len(node_id_to_index) != len(node_ids):
    raise ValueError("nodes.csv contains duplicate artwork_id values.")
artist_ids = [row["artist_id"] for row in nodes]
actual = np.asarray([int(row["label_id"]) for row in nodes], dtype=np.int64)

feature_rows = {}
feature_count = None
with (DATASET_DIR / "features.txt").open(encoding="utf-8") as source:
    for line_number, line in enumerate(source, start=1):
        fields = line.split()
        if len(fields) < 2:
            raise ValueError(f"Invalid feature row {line_number}: no feature values.")
        artwork_id = fields[0]
        if artwork_id not in node_id_to_index:
            raise ValueError(f"Feature row {line_number} references an unknown artwork_id.")
        if artwork_id in feature_rows:
            raise ValueError(f"Duplicate artwork_id in features.txt on line {line_number}.")
        if feature_count is None:
            feature_count = len(fields) - 1
        elif len(fields) - 1 != feature_count:
            raise ValueError(f"Inconsistent feature count on line {line_number}.")
        values = np.asarray(fields[1:], dtype=np.float32)
        if not np.isfinite(values).all():
            raise ValueError(f"Non-finite feature value on line {line_number}.")
        feature_rows[artwork_id] = values

if set(feature_rows) != set(node_ids):
    missing = len(set(node_ids) - set(feature_rows))
    raise ValueError(f"features.txt is missing {missing} artwork rows from nodes.csv.")
features = np.vstack([feature_rows[artwork_id] for artwork_id in node_ids])
if features.shape[0] < 2 or features.shape[1] == 0:
    raise ValueError("Need at least two artworks and one feature to build similarity graph.")

# Build a top-k cosine-neighbor graph while excluding same-artist links.
N_NEIGHBORS = 20
artist_counts = {}
for artist_id in artist_ids:
    artist_counts[artist_id] = artist_counts.get(artist_id, 0) + 1
query_count = min(
    len(nodes),
    N_NEIGHBORS + max(artist_counts.values()) + 1,
)
neighbors = NearestNeighbors(
    n_neighbors=query_count,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
).fit(features)
distances, indices = neighbors.kneighbors(features)
proposed_edges = {}
for source_index, (row_distances, row_indices) in enumerate(zip(distances, indices)):
    selected = 0
    for distance, target_index in zip(row_distances, row_indices):
        target_index = int(target_index)
        if target_index == source_index:
            continue
        if artist_ids[target_index] == artist_ids[source_index]:
            continue
        similarity = max(0.0, 1.0 - float(distance))
        if similarity <= 0.0:
            continue
        edge = (min(source_index, target_index), max(source_index, target_index))
        proposed_edges[edge] = max(similarity, proposed_edges.get(edge, 0.0))
        selected += 1
        if selected >= N_NEIGHBORS:
            break

graph = nx.Graph()
graph.add_nodes_from(range(len(nodes)))
for (source_index, target_index), weight in proposed_edges.items():
    graph.add_edge(source_index, target_index, weight=weight)
if graph.number_of_edges() == 0:
    raise ValueError("No positive cross-author feature similarities; Louvain cannot run.")

edge_path = OUTPUT_DIR / "artic_louvain_similarity_edges.csv"
with edge_path.open("w", encoding="utf-8-sig", newline="") as output:
    writer = csv.DictWriter(
        output,
        fieldnames=["source_node_index", "target_node_index", "source_artwork_id", "target_artwork_id", "cosine_similarity"],
    )
    writer.writeheader()
    for (source_index, target_index), weight in sorted(proposed_edges.items()):
        writer.writerow({
            "source_node_index": source_index,
            "target_node_index": target_index,
            "source_artwork_id": node_ids[source_index],
            "target_artwork_id": node_ids[target_index],
            "cosine_similarity": weight,
        })

graph_metadata = {
    "graph_source": "top_k_cosine_similarity_between_artwork_features",
    "features_file": "features.txt",
    "neighbors_per_artwork": N_NEIGHBORS,
    "same_artist_edges_excluded": True,
    "edge_symmetrization": "union; keep maximum cosine similarity",
    "nodes": graph.number_of_nodes(),
    "edges": graph.number_of_edges(),
    "connected_components": nx.number_connected_components(graph),
    "maximum_degree": max(dict(graph.degree()).values(), default=0),
    "mean_degree": float(np.mean([degree for _, degree in graph.degree()])),
    "cosine_similarity_min": min(data["weight"] for _, _, data in graph.edges(data=True)),
    "cosine_similarity_mean": float(np.mean([data["weight"] for _, _, data in graph.edges(data=True)])),
}
with (OUTPUT_DIR / "artic_louvain_graph_metadata.json").open("w", encoding="utf-8") as output:
    json.dump(graph_metadata, output, indent=2, ensure_ascii=False)

print(
    f"Built cross-author cosine graph: {graph.number_of_nodes():,} nodes, "
    f"{graph.number_of_edges():,} edges, "
    f"{graph_metadata['connected_components']:,} connected components, "
    f"max degree {graph_metadata['maximum_degree']}, "
    f"mean degree {graph_metadata['mean_degree']:.1f}."
)
print(f"Similarity edges: {edge_path}")
print("department_title is used only for external evaluation, not graph construction.")

In [ ]:
RUNS = 10
SEED = 42
RESOLUTION = 1.0


def clustering_accuracy(predicted, target):
    predicted = np.asarray(predicted, dtype=np.int64)
    target = np.asarray(target, dtype=np.int64)
    predicted_ids = np.unique(predicted)
    target_ids = np.unique(target)
    contingency = np.zeros((len(predicted_ids), len(target_ids)), dtype=np.int64)
    predicted_index = {label: index for index, label in enumerate(predicted_ids)}
    target_index = {label: index for index, label in enumerate(target_ids)}
    for pred, truth in zip(predicted, target):
        contingency[predicted_index[pred], target_index[truth]] += 1
    row_indices, column_indices = linear_sum_assignment(contingency.max() - contingency)
    return float(contingency[row_indices, column_indices].sum() / len(target))


run_metrics = []
for run_index in range(RUNS):
    seed = SEED + run_index
    start_time = time.perf_counter()
    communities = nx.community.louvain_communities(
        graph,
        weight="weight",
        resolution=RESOLUTION,
        seed=seed,
    )
    elapsed = time.perf_counter() - start_time

    community_by_node = {
        node_id: community_id
        for community_id, community in enumerate(communities)
        for node_id in community
    }
    predicted = np.asarray([community_by_node[node_id] for node_id in node_ids], dtype=np.int64)
    if len(predicted) != len(actual):
        raise ValueError("Louvain assignment count does not match number of artworks.")

    modularity = nx.community.modularity(
        graph,
        communities,
        weight="weight",
        resolution=RESOLUTION,
    )
    metrics = {
        "run": run_index + 1,
        "seed": seed,
        "resolution": RESOLUTION,
        "communities": len(communities),
        "graph_edges": graph.number_of_edges(),
        "graph_components": graph_metadata["connected_components"],
        "modularity": float(modularity),
        "ACC": clustering_accuracy(predicted, actual),
        "NMI": float(normalized_mutual_info_score(actual, predicted)),
        "ARI": float(adjusted_rand_score(actual, predicted)),
        "AMI": float(adjusted_mutual_info_score(actual, predicted)),
        "runtime_seconds": elapsed,
    }
    run_metrics.append(metrics)

    assignments_path = OUTPUT_DIR / f"artic_louvain_assignments_seed_{seed}.csv"
    with assignments_path.open("w", encoding="utf-8-sig", newline="") as output:
        writer = csv.DictWriter(
            output,
            fieldnames=[
                "node_index", "artwork_id", "artist_id", "title",
                "artist_title", "label_id", "department_title",
                "predicted_community",
            ],
        )
        writer.writeheader()
        for row, community_id in zip(nodes, predicted):
            writer.writerow({
                **{key: row.get(key, "") for key in writer.fieldnames[:-1]},
                "predicted_community": int(community_id),
            })

    print(
        f"Louvain {run_index + 1}/{RUNS} | seed={seed} | "
        f"communities={len(communities)} | modularity={modularity:.4f} | "
        f"NMI={metrics['NMI']:.4f} | ARI={metrics['ARI']:.4f} | "
        f"ACC={metrics['ACC']:.4f} | time={elapsed:.2f}s"
    )

    community_rows = []
    for community_id, community in enumerate(communities):
        member_indices = [index for index, node_id in enumerate(node_ids) if node_id in community]
        label_counts = {}
        artist_ids = set()
        for index in member_indices:
            label_id = int(nodes[index]["label_id"])
            department = nodes[index].get("department_title", "")
            label_counts[department] = label_counts.get(department, 0) + 1
            artist_ids.add(nodes[index].get("artist_id", ""))
        majority_department, majority_count = max(label_counts.items(), key=lambda item: item[1])
        community_rows.append({
            "community_id": community_id,
            "size": len(community),
            "artist_count": len(artist_ids),
            "majority_department": majority_department,
            "majority_department_share": majority_count / len(community),
            "department_counts": json.dumps(label_counts, ensure_ascii=False),
        })
    if run_index == 0:
        community_summary_path = OUTPUT_DIR / f"artic_louvain_communities_seed_{seed}.csv"
        with community_summary_path.open("w", encoding="utf-8-sig", newline="") as output:
            writer = csv.DictWriter(output, fieldnames=list(community_rows[0]))
            writer.writeheader()
            writer.writerows(community_rows)

metrics_path = OUTPUT_DIR / "artic_louvain_metrics.csv"
with metrics_path.open("w", encoding="utf-8-sig", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=list(run_metrics[0]))
    writer.writeheader()
    writer.writerows(run_metrics)

summary_metrics = (
    "communities", "modularity", "ACC", "NMI", "ARI", "AMI", "runtime_seconds"
)
summary_rows = []
summary_path = OUTPUT_DIR / "artic_louvain_metrics_summary.csv"
print("\n=== Louvain mean +/- sample standard deviation across seeds ===")
for metric in summary_metrics:
    values = np.asarray([row[metric] for row in run_metrics], dtype=float)
    mean = float(values.mean())
    std = float(values.std(ddof=1)) if len(values) > 1 else 0.0
    summary_rows.append({"metric": metric, "mean": mean, "std": std})
    print(f"{metric}: {mean:.4f} +/- {std:.4f}")
with summary_path.open("w", encoding="utf-8-sig", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=["metric", "mean", "std"])
    writer.writeheader()
    writer.writerows(summary_rows)

print(f"Metrics: {metrics_path}")
print(f"Summary: {summary_path}")
print(f"First-run community breakdown: {community_summary_path}")
print(f"Assignments: {OUTPUT_DIR / f'artic_louvain_assignments_seed_{SEED}.csv'}")